In [1]:
import requests
import pandas as pd
class MarketDataFetcher:
  def __init__(self):
    self.base_url = "https://api.coingecko.com/api/v3/coins/markets"
  
  def veri_cek(self, limit=100):
    response = requests.get(self.base_url,
                params= {"vs_currency": "usd",
                         "per_page": limit})
    raw_data = response.json()
    df = pd.DataFrame(raw_data)

    df = df[['id', 'symbol', 'current_price', 'market_cap', 'total_volume', 'price_change_percentage_24h']]

    return df

fetcher = MarketDataFetcher()
df = fetcher.veri_cek(limit=50)
print(df.head())



            id symbol  current_price     market_cap  total_volume  \
0      bitcoin    btc   68214.000000  1365740790793  5.346702e+10   
1     ethereum    eth    1977.900000   238999275424  2.506376e+10   
2       tether   usdt       0.999977   183746651417  8.628883e+10   
3  binancecoin    bnb     632.810000    86353213657  1.302429e+09   
4       ripple    xrp       1.360000    83178219902  2.761283e+09   

   price_change_percentage_24h  
0                     -1.54407  
1                     -3.39745  
2                     -0.01564  
3                     -0.86027  
4                     -2.76290  


In [2]:
class CryptoAnalyzer:
  def __init__(self,data_framework):
    self.df = data_framework
  
  def eksikleri_temizle(self):
    self.df = self.df.dropna()
    return self.df
  
  def ozet_istatistikler(self):
    print(f"Matematiksel ozet = {self.df.describe()}")

fetcher = MarketDataFetcher()
canli_veri = fetcher.veri_cek(limit=50)

analizci = CryptoAnalyzer(canli_veri)
analizci.eksikleri_temizle()
analizci.ozet_istatistikler()

Matematiksel ozet =        current_price    market_cap  total_volume  price_change_percentage_24h
count      50.000000  5.000000e+01  5.000000e+01                    50.000000
mean     1651.775790  4.614425e+10  3.835100e+09                    -1.624977
std      9660.984858  1.955838e+11  1.447723e+10                     2.665727
min         0.000005  1.622272e+09  0.000000e+00                   -10.253240
25%         0.693538  2.472759e+09  4.986696e+07                    -2.875175
50%         1.016500  4.201458e+09  1.948129e+08                    -0.924200
75%        44.870000  9.480477e+09  6.627267e+08                    -0.014943
max     68214.000000  1.365741e+12  8.628883e+10                     6.123510


In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import pandas as pd
class CryptoML:
  def __init__(self, clean_data):
    self.df = clean_data
    self.scaler = StandardScaler()
    self.model = KMeans(n_clusters=3,
                        random_state=42)

  def kumele(self):
      X = self.df[['price_change_percentage_24h',
                 'total_volume']]
      X_scaled = self.scaler.fit_transform(X)
      self.model.fit(X_scaled)
      predict = self.model.predict(X_scaled)
      self.df['Risk_Grubu'] = predict
      return self.df[['symbol',
                   'price_change_percentage_24h',
                   'Risk_Grubu']]

# 1. Veri Çekme
fetcher = MarketDataFetcher()
canli_veri = fetcher.veri_cek(limit=50)

# 2. Veri Temizleme ve EDA
analizci = CryptoAnalyzer(canli_veri)
analizci.df = analizci.eksikleri_temizle()

# 3. Makine Öğrenmesi (YENİ)
ml_motoru = CryptoML(analizci.df) # Analizciden çıkan temiz veriyi ML'e veriyoruz
sonuclar = ml_motoru.kumele()

print("\n🚀 YAPAY ZEKA KÜMELEME SONUÇLARI:")
print(sonuclar.head(15))


🚀 YAPAY ZEKA KÜMELEME SONUÇLARI:
        symbol  price_change_percentage_24h  Risk_Grubu
0          btc                     -1.54407           2
1          eth                     -3.39745           1
2         usdt                     -0.01564           2
3          bnb                     -0.86027           0
4          xrp                     -2.76290           1
5         usdc                     -0.01001           0
6          sol                     -2.17100           0
7          trx                     -0.80730           0
8   figr_heloc                      0.26486           0
9         doge                     -4.90901           1
10         wbt                     -1.98487           0
11        usds                     -0.02821           0
12         ada                     -6.14655           1
13         bch                     -1.22758           0
14         leo                      1.33910           0
